<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

## Point cloud loading / caching for E57 files exported from Dot3D.

Dot3D writes one E57 "scan" per captured frame (hundreds of small scans, each
with its own pose).  pye57 applies the pose when reading, so we simply
concatenate all scans into one XYZ / RGB array.

The loaded cloud is cached next to the E57 as an .npz so subsequent API calls
are fast (numpy load of a 1M point cloud is ~50 ms vs ~0.6 s for E57 parse;
for 50M+ point clouds the difference is minutes vs seconds).

In [4]:
#| echo: false
#| output: asis
show_doc(CloudInfo)

---

[source](https://github.com/owen-AR/ptsrv/blob/main/ptsrv/cloud.py#L27){target="_blank" style="float:right; font-size:smaller"}

### CloudInfo

```python
def CloudInfo(
    name:str, n_points:int, xmin:float, ymin:float, zmin:float, xmax:float, ymax:float, zmax:float, floor_z:float,
    ceiling_z:float, rotation_deg:float, source:str, extra:dict=<factory>
)->None:
```

*CloudInfo(name: 'str', n_points: 'int', xmin: 'float', ymin: 'float', zmin: 'float', xmax: 'float', ymax: 'float', zmax: 'float', floor_z: 'float', ceiling_z: 'float', rotation_deg: 'float', source: 'str', extra: 'dict' = <factory>)*

In [6]:
#| echo: false
#| output: asis
show_doc(Cloud)

---

[source](https://github.com/owen-AR/ptsrv/blob/main/ptsrv/cloud.py#L47){target="_blank" style="float:right; font-size:smaller"}

### Cloud

```python
def Cloud(
    xyz:np.ndarray, rgb:np.ndarray, info:CloudInfo
)->None:
```

*Cloud(xyz: 'np.ndarray', rgb: 'np.ndarray', info: 'CloudInfo')*

In [8]:
#| echo: false
#| output: asis
show_doc(read_e57)

---

[source](https://github.com/owen-AR/ptsrv/blob/main/ptsrv/cloud.py#L53){target="_blank" style="float:right; font-size:smaller"}

### read_e57

```python
def read_e57(
    path:str, max_points:int | None=None
)->tuple[np.ndarray, np.ndarray]:
```

*Read every scan in an E57 and return (xyz float32 (N,3), rgb uint8 (N,3)).*

Poses are applied by pye57.  If the file has no colour, rgb is mid-grey.
If max_points is set, the cloud is randomly subsampled to that size after
load (keeps memory bounded on a small server).

In [10]:
#| echo: false
#| output: asis
show_doc(detect_floor_ceiling)

---

[source](https://github.com/owen-AR/ptsrv/blob/main/ptsrv/cloud.py#L95){target="_blank" style="float:right; font-size:smaller"}

### detect_floor_ceiling

```python
def detect_floor_ceiling(
    z:np.ndarray, bin_size:float=0.02
)->tuple[float, float]:
```

*Find floor and ceiling as the two strongest horizontal planes in the Z histogram.*

Floor = strongest peak in the lower half of the Z range, ceiling = strongest
peak in the upper half.  Good enough for single storey interiors; for
multi storey clouds call with a Z-filtered array or set levels manually.

In [12]:
#| echo: false
#| output: asis
show_doc(detect_wall_rotation)

---

[source](https://github.com/owen-AR/ptsrv/blob/main/ptsrv/cloud.py#L114){target="_blank" style="float:right; font-size:smaller"}

### detect_wall_rotation

```python
def detect_wall_rotation(
    xyz:np.ndarray, floor_z:float, ceiling_z:float, cell:float=0.02
)->float:
```

*Estimate the rotation (deg, about Z) that aligns the dominant walls with X/Y.*

Takes a horizontal band of the cloud, rasterises it to a density image,
computes image gradients and finds the dominant gradient orientation
modulo 90 degrees.  Returns the angle to rotate the cloud by (counter-
clockwise positive).

In [14]:
#| echo: false
#| output: asis
show_doc(rotate_z)

---

[source](https://github.com/owen-AR/ptsrv/blob/main/ptsrv/cloud.py#L151){target="_blank" style="float:right; font-size:smaller"}

### rotate_z

```python
def rotate_z(
    xyz:np.ndarray, deg:float, about:tuple[float, float]=(0.0, 0.0)
)->np.ndarray:
```

In [17]:
#| echo: false
#| output: asis
show_doc(build_cloud)

---

[source](https://github.com/owen-AR/ptsrv/blob/main/ptsrv/cloud.py#L168){target="_blank" style="float:right; font-size:smaller"}

### build_cloud

```python
def build_cloud(
    e57_path:str, align:str | float='auto', max_points:int | None=None, name:str | None=None
)->Cloud:
```

*Read an E57, detect floor/ceiling, optionally align walls, return a Cloud.*

In [19]:
#| echo: false
#| output: asis
show_doc(load_cloud)

---

[source](https://github.com/owen-AR/ptsrv/blob/main/ptsrv/cloud.py#L191){target="_blank" style="float:right; font-size:smaller"}

### load_cloud

```python
def load_cloud(
    e57_path:str, align:str | float='auto', max_points:int | None=None, use_cache:bool=True
)->Cloud:
```

*Load with on-disk cache.  Cache is invalidated when the E57 mtime changes.*

In [21]:
#| echo: false
#| output: asis
show_doc(CloudCache)

---

[source](https://github.com/owen-AR/ptsrv/blob/main/ptsrv/cloud.py#L216){target="_blank" style="float:right; font-size:smaller"}

### CloudCache

```python
def CloudCache(
    max_items:int=2, **load_kwargs
):
```

*Small thread-safe in-memory LRU of loaded clouds for the API server.*